# ColabUMA Scan — UMAによる緩和スキャン

Metaの機械学習ポテンシャル **UMA** で、結合長・角度・二面角を一定の刻みで動かしながら残りの自由度を最適化する **緩和スキャン（relaxed scan）** を行うノートブックです。Gaussian の `opt=modredundant` の `S` 指定に相当します。コードを書く必要はありません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に名前 `HF_TOKEN`、値にHugging Faceのアクセストークンを登録し、「ノートブックからのアクセス」をオンにする
3. 下の **1. 入力フォーム** で出力名、電荷、スピン多重度、スキャン条件、収束判定を指定する
4. **ランタイム → すべて実行** を押す
5. **2. ファイル選択と事前確認** にファイル選択ボタンが出るので、xyz または Gaussian の gjf / com を選ぶ。あとは放置できます
6. 一番下のセルでエネルギープロファイルと各点の構造を確認し、結果のZIPをダウンロードする

原子番号が分からないときは、そのまま一度実行してください。「5. 構造の準備」に番号付きの3D表示が出ます。番号を確認してフォームを直し、もう一度実行します。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。

2回目以降は手順3〜6を繰り返すだけです（インストールとモデル読み込みは自動でスキップされます）。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## スキャン条件の指定

| 項目 | 説明 |
|---|---|
| `scan_atoms` | 原子番号（1始まり）をスペース区切りで。2個なら結合長、3個なら角度、4個なら二面角 |
| `scan_start` | 開始値（Å または 度）。空欄なら入力構造の値から始める |
| `scan_n_steps` | 刻みの回数。計算する点の数はこれに1を足した数になる（Gaussian の `S` と同じ） |
| `scan_step_size` | 1回の刻み幅（Å または 度）。負の値で逆向きに動かす |
| `opt_before_scan` | オンなら、スキャンの前に入力構造を拘束なしで最適化する。`scan_start` が空欄なら、最適化後の値から始める |
| `opt_after_scan` | オンなら、スキャンの後に拘束なしで最適化する。開始構造はスキャンの最後の点からさかのぼって最初に見つかる極小 |

たとえば Gaussian の `D 1 2 3 4 S 36 10.0` は、`scan_atoms = 1 2 3 4`、`scan_n_steps = 36`、`scan_step_size = 10` です。

「後の最適化」は、行き過ぎたスキャンの端から戻って生成物側の極小を取るためのものです。たとえば結合を縮めすぎてエネルギーが上がっていても、その上り坂をさかのぼった先の極小から最適化します。0.1 kcal/mol 未満の小さな凹凸と未収束の点は無視します。

## 出力（ZIPの中身）

| ファイル | 内容 |
|---|---|
| `<出力名>_scan.csv` | 各点の目標値、実測値、エネルギー、ΔE、fmax、ステップ数、収束（未収束の点も載せます） |
| `<出力名>_scan.xyz` | 収束した各点の最適化構造（複数構造xyz） |
| `<出力名>_scan.log` | 収束した各点の最終構造だけを Gaussian のスキャン出力の書式で書いたもの。Gaussian で計算した結果ではありません |
| `<出力名>_preopt.xyz` / `.log` | 前の最適化の結果と経路（オンのとき） |
| `<出力名>_postopt.xyz` / `.log` | 後の最適化の結果と経路（オンのとき） |
| `<出力名>_log.txt` | 入力条件、ソフトのバージョン、各点と各ステップのエネルギーと fmax |

## 注意事項

- 各点は前の点の最適化構造から始めます。途中で構造が別の谷に飛ぶと、プロファイルに段差が出ます。逆向きにもスキャンして、同じ曲線になるか確認してください。
- プロファイルの最高点は遷移状態ではありません。遷移状態の初期構造の候補として使い、別途最適化して確認してください。
- 電荷とスピン多重度は全点で固定です。結合を切るスキャンでは、解離の極限でエネルギーが正しくならないことがあります。
- 環の中の座標をスキャンすると、指定の末端の原子だけを動かすため、各点の初期構造が大きくひずむことがあります。刻み幅を小さくしてください。
- UMAは機械学習ポテンシャルであり、DFTの代わりにはなりません。最終的なエネルギー比較はDFTで確認してください。
- 周期系（結晶・表面）には対応していません。

## ライセンス

- このノートのコード：MIT License
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

In [ ]:
#@title 1. 入力フォーム { display-mode: "form" }
#@markdown 入力が終わったら **ランタイム → すべて実行** を押してください。次のセルで xyz または gjf ファイルを選ぶと、あとは放置できます。

#@markdown ### 出力
output_name = "result"  #@param {type:"string"}

#@markdown ### 構造
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### スキャン条件
#@markdown 原子番号（1始まり）をスペース区切りで指定します。2個なら結合長、3個なら角度、4個なら二面角です。
scan_atoms = "1 2 3 4"  #@param {type:"string"}
#@markdown 開始値（Å または 度）。空欄なら入力構造の値から始めます（前の最適化がオンなら最適化後の値）。
scan_start = ""  #@param {type:"string"}
#@markdown 刻みの回数と刻み幅（Å または 度）。計算する点の数は、刻みの回数に1を足した数です。
scan_n_steps = 36  #@param {type:"integer"}
scan_step_size = 10.0  #@param {type:"number"}
#@markdown スキャンの前に、入力構造を拘束なしで最適化します。
opt_before_scan = True  #@param {type:"boolean"}
#@markdown スキャンの後に、最後の点からさかのぼって最初の極小を拘束なしで最適化します。
opt_after_scan = True  #@param {type:"boolean"}

#@markdown ### 計算条件
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown 収束判定 fmax（eV/Å）。normal は Gaussian の既定、tight は Gaussian の opt=tight と同じくらいです。
convergence = "normal（0.02）"  #@param ["loose（0.05）", "normal（0.02）", "tight（0.001）", "数値指定"]
#@markdown 「数値指定」を選んだときの fmax（eV/Å）。
fmax_custom = 0.02  #@param {type:"number"}
#@markdown 最大ステップ数は1点あたりの値です。
max_steps = 500  #@param {type:"integer"}

In [ ]:
#@title 2. ファイル選択と事前確認 { display-mode: "form" }
#@markdown ファイルの選択とトークンの確認をここで済ませます。このあとのインストールと計算は放置できます。
import html
import os
import re
import shutil
import subprocess

from IPython import get_ipython
from IPython.display import HTML, display


class ColabUMAError(Exception):
    # 利用者に見せるエラー。トレースバックの代わりに日本語の一文と対処法を表示する
    def __init__(self, message, hint=""):
        super().__init__(message)
        self.message = message
        self.hint = hint


def _box(kind, message, hint=""):
    bg, fg, label = {
        "error": ("#fdecea", "#b3261e", "エラー"),
        "warn": ("#fff4e5", "#8a5300", "注意"),
        "ok": ("#e8f5e9", "#1b5e20", "OK"),
    }[kind]
    hint_html = f"<div style='margin-top:4px'>対処法：{html.escape(hint)}</div>" if hint else ""
    display(HTML(
        f"<div style='background:{bg};color:{fg};padding:10px 14px;border-radius:6px;"
        f"margin:6px 0;font-size:14px;line-height:1.6'><b>{label}</b>　{html.escape(message)}{hint_html}</div>"
    ))


def warn(message, hint=""):
    _box("warn", message, hint)


def ok(message):
    _box("ok", message)


def _show_colabuma_error(shell, etype, evalue, tb, tb_offset=None):
    _box("error", evalue.message, evalue.hint)
    return []


get_ipython().set_custom_exc((ColabUMAError,), _show_colabuma_error)

# --- フォームの値の確認 ---
FMAX_PRESETS = {"loose（0.05）": 0.05, "normal（0.02）": 0.02, "tight（0.001）": 0.001}
fmax = fmax_custom if convergence == "数値指定" else FMAX_PRESETS[convergence]
if fmax <= 0:
    raise ColabUMAError("収束判定 fmax は正の数にしてください。", "通常は 0.02 eV/Å 程度です。")
if max_steps < 1:
    raise ColabUMAError("最大ステップ数は1以上にしてください。", "通常は 500 程度です。")
if multiplicity < 1:
    raise ColabUMAError("スピン多重度は1以上の整数にしてください。", "閉殻分子なら1（一重項）です。")
out_prefix = re.sub(r"[^\w\-.]", "_", output_name.strip()) or "result"

# --- スキャン条件の確認（原子数が要らないものはここで調べる） ---
_atoms_hint = "2個なら結合長、3個なら角度、4個なら二面角です（例：1 2 3 4）。"
try:
    scan_idx = [int(t) - 1 for t in scan_atoms.replace(",", " ").split()]
except ValueError:
    raise ColabUMAError(f"scan_atoms「{scan_atoms}」を原子番号として読み取れませんでした。", _atoms_hint) from None
if len(scan_idx) not in (2, 3, 4):
    raise ColabUMAError(f"scan_atoms には原子番号を2〜4個指定してください（今は {len(scan_idx)} 個です）。", _atoms_hint)
if min(scan_idx) < 0:
    raise ColabUMAError(f"scan_atoms の原子番号は1以上で指定してください（今は {scan_atoms} です）。", _atoms_hint)
if len(set(scan_idx)) != len(scan_idx):
    raise ColabUMAError(f"scan_atoms に同じ原子番号が入っています（{scan_atoms}）。", _atoms_hint)
if scan_n_steps < 1:
    raise ColabUMAError("刻みの回数は1以上にしてください。", "計算する点の数は、刻みの回数に1を足した数です。")
if scan_step_size == 0:
    raise ColabUMAError("刻み幅に0は指定できません。", "結合長なら 0.1 Å、角度や二面角なら 10 度 程度が目安です。")
scan_start_value = None
if scan_start.strip():
    try:
        scan_start_value = float(scan_start)
    except ValueError:
        raise ColabUMAError(f"scan_start「{scan_start}」を数値として読み取れませんでした。",
                            "空欄にすると入力構造の値から始めます。") from None
if len(scan_idx) == 4 and abs(scan_n_steps * scan_step_size) > 360.0 + 1e-6:
    warn("二面角を1回転より多く動かす設定です。同じ角度を重複して計算します。")

# --- GPUの確認（torchを読み込む前に nvidia-smi で調べる） ---
_has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi"], capture_output=True).returncode == 0
if not _has_gpu:
    warn("GPUが見つからないため、CPUで計算します。計算はかなり遅くなります。",
         "メニューの「ランタイム → ランタイムのタイプを変更」で「T4 GPU」を選び、もう一度すべて実行してください。")

# --- Hugging Faceのトークン（ノートに保存しない。検証はインストール後に行う） ---
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        _token = userdata.get("HF_TOKEN")
    except Exception:
        _token = None
    if not _token:
        warn("ColabのシークレットにHF_TOKENが登録されていません。",
             "左サイドバーの鍵アイコン（シークレット）で、名前 HF_TOKEN・値にトークンを登録し、"
             "「ノートブックからのアクセス」をオンにしてください。今回は下の欄に貼り付けても続行できます。")
        from getpass import getpass
        _token = getpass("Hugging Faceのアクセストークンを貼り付けてEnter: ").strip()
    if not _token:
        raise ColabUMAError("Hugging Faceのアクセストークンがありません。",
                            "https://huggingface.co/settings/tokens でRead権限のトークンを作り、シークレットにHF_TOKENとして登録してください。")
    os.environ["HF_TOKEN"] = _token
    del _token

# --- ファイル選択 ---
GJF_EXTENSIONS = (".gjf", ".com", ".gjc")
from google.colab import files

_previous = globals().get("upload_name")
print("xyz または gjf（.gjf / .com）ファイルを選んでください。")
if _previous:
    print(f"前回の「{_previous}」をそのまま使うときは Cancel upload を押してください。")
uploaded = files.upload()
if uploaded:
    if len(uploaded) > 1:
        warn("複数のファイルが選ばれたため、最初の1つだけを使います。")
    upload_name, upload_data = next(iter(uploaded.items()))
elif _previous:
    print(f"前回のファイル「{_previous}」を使います。")
else:
    raise ColabUMAError("ファイルがアップロードされませんでした。", "もう一度すべて実行して、xyz または gjf ファイルを選んでください。")


def _quick_atom_count(name, data):
    # インストール前に scan_atoms の番号の範囲だけ調べる。読めなければ None（あとで詳しく調べる）
    text = data.decode("utf-8", "replace").strip()
    if os.path.splitext(name)[1].lower() in GJF_EXTENSIONS:
        blocks = re.split(r"\n\s*\n", text.replace("\r\n", "\n"))
        return len(blocks[2].splitlines()) - 1 if len(blocks) >= 3 else None
    try:
        return int(text.split()[0])
    except (IndexError, ValueError):
        return None


_n_atoms = _quick_atom_count(upload_name, upload_data)
if _n_atoms and max(scan_idx) >= _n_atoms:
    raise ColabUMAError(f"scan_atoms の原子番号は1〜{_n_atoms}の範囲で指定してください（今は {scan_atoms} です）。",
                        "フォームを直して、もう一度すべて実行してください。ファイル選択は Cancel upload で前回のファイルを使えます。")

ok("入力の確認が終わりました。ここからは放置できます。")

In [ ]:
#@title 3. セットアップ（インストール・認証） { display-mode: "form" }
FAIRCHEM_VERSION = "2.25.0"

import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as _pkg_version


def _installed(pkg):
    try:
        return _pkg_version(pkg)
    except PackageNotFoundError:
        return None


# --- ライブラリのインストール（インストール済みならスキップ） ---
if _installed("fairchem-core") != FAIRCHEM_VERSION or not _installed("py3Dmol"):
    _torch_loaded_before = "torch" in sys.modules
    print(f"ライブラリをインストールしています（fairchem-core {FAIRCHEM_VERSION}）。数分かかります…")
    _r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"fairchem-core=={FAIRCHEM_VERSION}", "py3Dmol"],
        capture_output=True, text=True)
    if _r.returncode != 0:
        print("\n".join(_r.stderr.strip().splitlines()[-15:]))
        raise ColabUMAError("ライブラリのインストールに失敗しました（上にpipのメッセージを表示しています）。",
                            "「ランタイム → セッションを再起動」してから、もう一度すべて実行してください。")
    if _torch_loaded_before:
        raise ColabUMAError("インストールは完了しましたが、読み込み済みのライブラリを入れ替えるためセッションの再起動が必要です。",
                            "「ランタイム → セッションを再起動」してから、もう一度すべて実行してください。")
    ok("インストールが完了しました。")
else:
    print(f"fairchem-core {FAIRCHEM_VERSION} はインストール済みです。スキップします。")

# --- Hugging Faceの認証（トークンはノートに保存しない） ---
if not globals().get("_COLABUMA_AUTH_OK"):
    from huggingface_hub import whoami
    from huggingface_hub.utils import GatedRepoError, HfHubHTTPError, RepositoryNotFoundError
    try:
        whoami()
    except Exception as e:
        if getattr(getattr(e, "response", None), "status_code", None) == 401:
            raise ColabUMAError("Hugging Faceのトークンが無効です。",
                                "https://huggingface.co/settings/tokens でトークンを作り直し、シークレットのHF_TOKENを更新してください。") from None
        raise ColabUMAError(f"Hugging Faceに接続できませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    try:
        from huggingface_hub import auth_check
        auth_check("facebook/UMA")
    except ImportError:
        pass  # 古いhuggingface_hubでは確認を省略し、モデル読み込み時に判定する
    except GatedRepoError:
        raise ColabUMAError("facebook/UMA を使う権限がまだありません。",
                            "https://huggingface.co/facebook/UMA でアクセス申請してライセンスに同意し、承認されてからもう一度実行してください。") from None
    except RepositoryNotFoundError:
        raise ColabUMAError("Hugging Faceのトークンが無効です。",
                            "トークンを作り直し、シークレットのHF_TOKENを更新してください。") from None
    except HfHubHTTPError as e:
        raise ColabUMAError(f"Hugging Faceに接続できませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    _COLABUMA_AUTH_OK = True
    ok("Hugging Faceの認証とfacebook/UMAへのアクセス権を確認しました。")

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("計算デバイス:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

In [ ]:
#@title 4. モデル読み込み { display-mode: "form" }
import gc

from fairchem.core import FAIRChemCalculator, pretrained_mlip
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

_key = (model_name, DEVICE)
if globals().get("_UMA_KEY") != _key:
    globals().pop("_UMA_PREDICTOR", None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"{model_name} を読み込んでいます（初回はダウンロードに数分かかります）…")
    try:
        _UMA_PREDICTOR = pretrained_mlip.get_predict_unit(model_name, device=DEVICE)
    except GatedRepoError:
        raise ColabUMAError("facebook/UMA を使う権限がまだありません。",
                            "https://huggingface.co/facebook/UMA でアクセス申請してライセンスに同意し、承認されてからもう一度実行してください。") from None
    except HfHubHTTPError as e:
        _status = getattr(getattr(e, "response", None), "status_code", None)
        if _status in (401, 403):
            raise ColabUMAError("モデルのダウンロードが拒否されました（アクセス権がありません）。",
                                "https://huggingface.co/facebook/UMA でアクセス申請し、トークンが正しいか確認してください。") from None
        raise ColabUMAError(f"モデルをダウンロードできませんでした（{e}）。",
                            "しばらく待ってからもう一度実行してください。") from None
    except KeyError:
        raise ColabUMAError(f"モデル名「{model_name}」はこのバージョンのfairchemにありません。",
                            "フォームで別のモデルを選んでください。") from None
    _UMA_KEY = _key
    ok(f"{model_name} を読み込みました。")
else:
    print(f"{model_name} は読み込み済みです。スキップします。")

calc = FAIRChemCalculator(_UMA_PREDICTOR, task_name=task_name)

In [ ]:
#@title 5. 構造の準備 { display-mode: "form" }
import re
import tempfile

import numpy as np
from ase import Atoms
from ase.data import chemical_symbols, covalent_radii
from ase.io import read as ase_read

MAX_Z = 83  # UMAのomolタスク（OMol25）が扱える元素は H〜Bi

def check_elements(numbers):
    bad = sorted({chemical_symbols[z] if 0 < z < len(chemical_symbols) else "X"
                  for z in numbers if z < 1 or z > MAX_Z})
    if bad:
        raise ColabUMAError(f"UMAが扱えない元素が含まれています：{', '.join(bad)}",
                            "UMA（omol）が扱えるのは H〜Bi（原子番号1〜83）です。該当する元素を除くか、別の手法を使ってください。")


_FLOAT = re.compile(r"^[-+]?(\d+\.?\d*|\.\d+)([eEdD][-+]?\d+)?$")


def _gjf_cartesian_line(line):
    # 「元素 [凍結フラグ] x y z [ONIOMの層など]」を (元素記号, xyz) にする。読めなければ None
    tok = line.replace(",", " ").split()
    m = re.match(r"[A-Za-z]+|\d+", tok[0])  # "C(Fragment=1)", "C-CA--0.1", "C12" → "C"、"6" → 原子番号
    if not m:
        return None
    label = m.group(0)
    if label.isdigit():
        z = int(label)
        sym = chemical_symbols[z] if 0 < z < len(chemical_symbols) else label
    elif label.lower().startswith("bq") or label.upper() == "X":
        sym = "X"  # ゴースト原子・ダミー原子（後でエラーにする）
    elif label.lower().startswith("tv"):
        sym = "Tv"
    else:
        sym = label[:2].capitalize() if label[:2].capitalize() in chemical_symbols else label[:1].upper()
    nums = tok[1:]
    is_f = [bool(_FLOAT.match(t)) for t in nums]
    if len(nums) >= 4 and re.fullmatch(r"-?\d+", nums[0]) and all(is_f[1:4]) and (len(nums) == 4 or not is_f[4]):
        xyz = nums[1:4]
    elif len(nums) >= 3 and all(is_f[:3]) and (len(nums) == 3 or not is_f[3]):
        xyz = nums[:3]
    else:
        return None
    return tok[0], sym, [float(t.replace("D", "E").replace("d", "e")) for t in xyz]


def read_gjf(path, fname):
    # Gaussianの入力ファイルから構造だけを読む。電荷・多重度は確認用に返すだけで、計算にはフォームの値を使う
    with open(path, encoding="utf-8", errors="replace") as f:
        text = re.split(r"(?im)^\s*--link1--\s*$", f.read())[0]
    lines = [l.split("!")[0].rstrip() for l in text.splitlines()]
    i, n = 0, len(lines)
    while i < n and (not lines[i].strip() or lines[i].lstrip().startswith("%")):
        i += 1
    if i >= n or not lines[i].lstrip().startswith("#"):
        raise ColabUMAError(f"「{fname}」にルートセクション（# で始まる行）が見つかりませんでした。",
                            "Gaussianの入力ファイル（.gjf / .com）か確認してください。")
    for _ in range(2):  # ルートセクションとタイトルを読み飛ばす
        while i < n and lines[i].strip():
            i += 1
        while i < n and not lines[i].strip():
            i += 1
    cm = lines[i].replace(",", " ").split() if i < n else []
    try:
        gjf_cm = (int(cm[0]), int(cm[1]))
    except (IndexError, ValueError):
        raise ColabUMAError(f"「{fname}」の電荷・多重度の行を読み取れませんでした。",
                            "ルートセクションとタイトルの後に空行があり、その次の行が「0 1」のような電荷・多重度になっているか確認してください。") from None
    i += 1
    atom_lines = []
    while i < n and lines[i].strip():
        atom_lines.append(lines[i])
        i += 1
    parsed = [_gjf_cartesian_line(l) for l in atom_lines]
    if atom_lines and all(parsed):
        for raw, sym, _ in parsed:
            if sym == "Tv":
                raise ColabUMAError("gjfに並進ベクトル（Tv）があります。周期系の最適化には対応していません。",
                                    "Tv の行を消して、分子として計算してください。")
            if sym not in chemical_symbols or sym == "X":
                raise ColabUMAError(f"gjfの原子「{raw}」を元素として読み取れませんでした。",
                                    "ゴースト原子（Bq）やダミー原子（X）は使えません。その行を消してください。")
        return Atoms([p[1] for p in parsed], positions=[p[2] for p in parsed]), gjf_cm
    try:  # デカルト座標でなければ Z-matrix とみなして ASE で読む
        atoms = ase_read(path, format="gaussian-in")
        atoms.calc = None
        return atoms, gjf_cm
    except Exception:
        raise ColabUMAError(f"「{fname}」の分子構造を読み取れませんでした。",
                            "GaussViewで開いて保存し直すか、「元素記号 x y z」のデカルト座標で書かれているか確認してください。") from None


# --- 初期構造の読み込み ---
fname, data = upload_name, upload_data
path = os.path.join(tempfile.mkdtemp(), os.path.basename(fname))
with open(path, "wb") as f:
    f.write(data)
if os.path.splitext(fname)[1].lower() in GJF_EXTENSIONS:
    atoms0, gjf_cm = read_gjf(path, fname)
    if gjf_cm != (charge, multiplicity):
        warn(f"gjfに書かれた電荷・多重度（{gjf_cm[0]} {gjf_cm[1]}）とフォームの値（{charge} {multiplicity}）が違います。"
             "計算にはフォームの値を使います。")
    kind = "gjf"
else:
    frames = None
    for fmt in ("extxyz", "xyz"):
        try:
            frames = ase_read(path, index=":", format=fmt)
            break
        except Exception:
            pass
    if not frames:
        raise ColabUMAError(f"「{fname}」をxyzファイルとして読み込めませんでした。",
                            "1行目が原子数、2行目がコメント、3行目以降が「元素記号 x y z」（単位Å）の形式か確認してください。"
                            "Gaussianの入力ファイルなら拡張子を .gjf か .com にしてください。")
    if len(frames) > 1:
        warn(f"「{fname}」には {len(frames)} 個の構造が入っています。先頭の構造だけを使います。")
    atoms0 = frames[0]
    kind = "xyz"
atoms0.set_pbc(False)
atoms0.set_cell(np.zeros((3, 3)))
atoms0.info = {}
source_desc = f"{kind}: {fname}"

# --- 元素・電荷・多重度の確認 ---
numbers = atoms0.numbers
check_elements(numbers)
n_electrons = int(numbers.sum()) - int(charge)
n_unpaired = int(multiplicity) - 1
if n_electrons <= 0 or n_unpaired > n_electrons or (n_electrons - n_unpaired) % 2:
    suggest = "1, 3, …" if n_electrons % 2 == 0 else "2, 4, …"
    raise ColabUMAError(
        f"電荷 {charge:+d}・スピン多重度 {multiplicity} の組み合わせはこの分子では成り立ちません（電子数 {n_electrons}）。",
        f"多重度は 2S+1 です。電子数が偶数なら多重度は奇数、奇数なら偶数になります。この電荷なら多重度は {suggest} が候補です。")
atoms0.info.update({"charge": int(charge), "spin": int(multiplicity)})

ok(f"{atoms0.get_chemical_formula()}（{len(atoms0)} 原子）の初期構造を用意しました。")


# --- 番号付きの3D表示 ---
import py3Dmol

SCAN_KINDS = {2: ("R", "結合長", "Å", "Distance (Å)"), 3: ("A", "角度", "度", "Angle (deg)"),
              4: ("D", "二面角", "度", "Dihedral (deg)")}


def xyz_block(symbols, positions, comment):
    lines = [str(len(symbols)), comment]
    lines += [f"{s:<2s} {x:16.10f} {y:16.10f} {z:16.10f}" for s, (x, y, z) in zip(symbols, positions)]
    return "\n".join(lines) + "\n"


_highlight = {str(i + 1) for i in scan_idx}
_view = py3Dmol.view(width=640, height=420)
_view.addModel(xyz_block(atoms0.get_chemical_symbols(), atoms0.positions, ""), "xyz")
_view.setStyle({"stick": {"radius": 0.12}, "sphere": {"scale": 0.2}})
for _i, (_x, _y, _z) in enumerate(atoms0.positions):
    _view.addLabel(str(_i + 1), {"position": {"x": float(_x), "y": float(_y), "z": float(_z)}, "fontSize": 11,
                                 "fontColor": "white", "backgroundOpacity": 0.75,
                                 "backgroundColor": "#d62728" if str(_i + 1) in _highlight else "#555555"})
_view.setBackgroundColor("white")
_view.zoomTo()
print("原子番号（赤がスキャンに指定した原子です）：")
_view.show()

# --- スキャン座標の確認 ---
if any(i >= len(atoms0) for i in scan_idx):
    raise ColabUMAError(f"scan_atoms の原子番号は1〜{len(atoms0)}の範囲で指定してください（今は {scan_atoms} です）。",
                        "上の3D表示で番号を確認してください。")

scan_code, scan_name, scan_unit, scan_xlabel = SCAN_KINDS[len(scan_idx)]
_collinear_hint = "原子の指定を変えてください。直線に近い原子の組では、角度や二面角を決められません。"


def check_scan_geometry(atoms):
    if scan_code == "A" and not 1.0 < atoms.get_angle(*scan_idx) < 179.0:
        raise ColabUMAError(f"指定した3原子がほぼ直線上に並んでいます（{atoms.get_angle(*scan_idx):.1f} 度）。", _collinear_hint)
    if scan_code == "D":
        for tri in (scan_idx[:3], scan_idx[1:]):
            if not 5.0 < atoms.get_angle(*tri) < 175.0:
                raise ColabUMAError(f"二面角を作る3原子（{' '.join(str(i + 1) for i in tri)}）がほぼ直線上に並んでいます。", _collinear_hint)


def wrap180(v):
    # 二面角を (-180, 180] に収める
    return 180.0 - (180.0 - v) % 360.0


def scan_value(atoms):
    if scan_code == "R":
        return float(atoms.get_distance(*scan_idx))
    if scan_code == "A":
        return float(atoms.get_angle(*scan_idx))
    return float(wrap180(atoms.get_dihedral(*scan_idx)))


def make_targets(atoms):
    start = scan_value(atoms) if scan_start_value is None else scan_start_value
    targets = [start + k * scan_step_size for k in range(scan_n_steps + 1)]
    if scan_code == "R" and min(targets) < 0.3:
        raise ColabUMAError(f"結合長が {min(targets):.3f} Å まで短くなる設定です。", "開始値、刻みの回数、刻み幅を見直してください。")
    if scan_code == "A" and not (0.0 < min(targets) and max(targets) < 180.0):
        raise ColabUMAError(f"角度が {min(targets):.1f}〜{max(targets):.1f} 度の範囲を動く設定です。",
                            "角度は0度と180度を含まない範囲で指定してください。180度ちょうどでは拘束を定義できません。")
    return targets


check_scan_geometry(atoms0)
scan_targets = make_targets(atoms0)

# --- 座標を動かすときに一緒に動かす原子（入力構造の結合で決める） ---
_radii = covalent_radii[atoms0.numbers]
_bonded = atoms0.get_all_distances() < 1.25 * (_radii[:, None] + _radii[None, :])
np.fill_diagonal(_bonded, False)


def _reach(start, blocked):
    seen, stack = {start}, [start]
    while stack:
        for j in np.flatnonzero(_bonded[stack.pop()]):
            if int(j) not in seen and int(j) not in blocked:
                seen.add(int(j))
                stack.append(int(j))
    return seen


def moving_atoms(idx):
    # 動かす側の断片を返す。環の中で切り離せないときは末端の原子だけを動かす
    pivot, start, moved = (idx[0], idx[1], idx[1]) if len(idx) == 2 else (idx[1], idx[2], idx[-1])
    side = _reach(start, {pivot})
    if moved not in side:
        side = _reach(moved, {pivot, start})
    fixed_side = (set(int(j) for j in np.flatnonzero(_bonded[pivot])) | {idx[0]}) - {start, moved}
    if side & fixed_side:
        return [moved], True
    return sorted(side), False


scan_moving, _in_ring = moving_atoms(scan_idx)
if _in_ring:
    warn(f"指定した{scan_name}は環の中にあるため、原子 {scan_idx[-1] + 1} だけを動かして各点を始めます。"
         "各点の初期構造が大きくひずむことがあります。",
         "エネルギーが不連続になるときは、刻み幅を小さくしてください。")

ok(f"{scan_name} {' '.join(str(i + 1) for i in scan_idx)} を "
   f"{len(scan_targets)} 点スキャンします（入力構造の値 {scan_value(atoms0):.3f} {scan_unit}）。")

In [ ]:
#@title 6. スキャン { display-mode: "form" }
import time

import ase.units
from ase.constraints import FixInternals
from ase.optimize import BFGS, FIRE, LBFGS

OPTIMIZERS = {"LBFGS": LBFGS, "BFGS": BFGS, "FIRE": FIRE}
SCAN_TOLERANCE = 0.01  # Å または 度。目標値と実測値の差がこれを超えたら注意を出す
MIN_PROMINENCE = 0.1  # kcal/mol。極小を探すとき、これより小さい凹凸は無視する


def _fmax_of(forces):
    return float(np.sqrt((forces ** 2).sum(axis=1).max()))


def drive(atoms, target):
    # 動かす側の断片ごと、スキャン座標を目標値まで動かす
    if scan_code == "R":
        a, b = scan_idx
        v = atoms.positions[b] - atoms.positions[a]
        d = np.linalg.norm(v)
        atoms.positions[scan_moving] += v / d * (target - d)
    elif scan_code == "A":
        atoms.set_angle(*scan_idx, target, indices=scan_moving)
    else:
        atoms.set_dihedral(*scan_idx, target, indices=scan_moving)


def scan_constraint(target):
    if scan_code == "R":
        return FixInternals(bonds=[[target, list(scan_idx)]])
    if scan_code == "A":
        return FixInternals(angles_deg=[[target, list(scan_idx)]])
    return FixInternals(dihedrals_deg=[[wrap180(target), list(scan_idx)]])


def run_optimizer(atoms):
    atoms.calc = calc
    hist = {"energy": [], "fmax": [], "frames": [], "forces": []}
    opt = OPTIMIZERS[optimizer_name](atoms, logfile=None)

    def record():
        e = float(atoms.get_potential_energy())
        forces = atoms.get_forces()  # 拘束があれば、拘束方向の成分を除いた力
        hist["energy"].append(e)
        hist["fmax"].append(_fmax_of(forces))
        hist["frames"].append(atoms.positions.copy())
        hist["forces"].append(forces.copy())

    opt.attach(record, interval=1)
    t0 = time.time()
    try:
        opt.run(fmax=fmax, steps=max_steps)
    except torch.cuda.OutOfMemoryError:
        raise ColabUMAError("GPUのメモリが足りません。",
                            "モデルを uma-s-1p1 などの小さいものにするか、原子数を減らしてください。") from None
    return {"atoms": atoms, "energy": hist["energy"][-1], "fmax": hist["fmax"][-1], "steps": opt.nsteps,
            "converged": hist["fmax"][-1] < fmax, "hist": hist, "seconds": time.time() - t0}


def optimize_point(start_atoms, target, k):
    atoms = start_atoms.copy()
    atoms.set_constraint()
    drive(atoms, target)
    atoms.set_constraint(scan_constraint(target))
    r = run_optimizer(atoms)
    value = scan_value(atoms)
    deviation = wrap180(value - target) if scan_code == "D" else value - target
    print(f"  点 {k + 1:3d}/{len(scan_targets)}: {scan_name} = {target:9.3f} {scan_unit}  E = {r['energy']:.6f} eV  "
          f"fmax = {r['fmax']:.4f} eV/Å  {r['steps']:4d} ステップ  {'収束' if r['converged'] else '未収束'}"
          f"（{r['seconds']:.1f} 秒）")
    r.update({"point": k + 1, "target": target, "value": target + deviation, "deviation": deviation})
    return r


def free_opt(start_atoms, label):
    atoms = start_atoms.copy()
    atoms.set_constraint()
    r = run_optimizer(atoms)
    r.update({"label": label, "value": scan_value(atoms)})
    print(f"  {label}: {scan_name} = {r['value']:.3f} {scan_unit}  E = {r['energy']:.6f} eV  fmax = {r['fmax']:.4f} eV/Å  "
          f"{r['steps']:4d} ステップ  {'収束' if r['converged'] else '未収束'}（{r['seconds']:.1f} 秒）")
    return r


def last_minimum_index(energies, converged):
    # 最後の点からさかのぼって最初の極小の番号を返す。未収束の点と小さな凹凸は無視する
    tol = MIN_PROMINENCE * ase.units.kcal / ase.units.mol
    usable = [i for i, c in enumerate(converged) if c] or list(range(len(energies)))
    best = usable[-1]
    for i in reversed(usable[:-1]):
        if energies[i] < energies[best]:
            best = i
        elif energies[i] - energies[best] > tol:
            break
    return best


# --- 前の最適化（オンのとき） ---
pre_result = None
_current = atoms0
if opt_before_scan:
    print(f"スキャンの前に {optimizer_name} で最適化します（fmax < {fmax} eV/Å、最大 {max_steps} ステップ）")
    pre_result = free_opt(atoms0, "前の最適化")
    _current = pre_result["atoms"]
    check_scan_geometry(_current)
    scan_targets = make_targets(_current)

# --- スキャン ---
print(f"{optimizer_name} で各点を最適化します（fmax < {fmax} eV/Å、1点あたり最大 {max_steps} ステップ）")
scan_results = []
for _k, _target in enumerate(scan_targets):
    _r = optimize_point(_current, _target, _k)
    scan_results.append(_r)
    _current = _r["atoms"]
_usable = [r for r in scan_results if r["converged"]] or scan_results  # 未収束の点のエネルギーは基準にしない
lowest = min(_usable, key=lambda r: r["energy"])
highest = max(_usable, key=lambda r: r["energy"])

# --- 後の最適化（オンのとき） ---
post_result = None
post_index = None
if opt_after_scan:
    post_index = last_minimum_index([r["energy"] for r in scan_results], [r["converged"] for r in scan_results])
    print(f"最後の点からさかのぼって最初の極小は点 {post_index + 1} です。ここから拘束なしで最適化します。")
    post_result = free_opt(scan_results[post_index]["atoms"], "後の最適化")

In [ ]:
#@title 7. 結果表示と保存 { display-mode: "form" }
import datetime
import platform
import zipfile

import ase
import ase.units
import matplotlib.pyplot as plt
import pandas as pd

EV_PER_HARTREE = ase.units.Hartree
EV_PER_KCALMOL = ase.units.kcal / ase.units.mol


def _gaussian_header(route, first):
    return [
        " Entering Gaussian System, Link 0=g16",
        " ******************************************",
        " Output format compatible with Gaussian 16 (Gaussian, Inc.) so that GaussView and other readers can open it.",
        " This file was written by ColabUMA Scan. It is NOT the output of a Gaussian calculation:",
        f" energies and forces come from the UMA machine-learning potential ({model_name}, task={task_name}).",
        " ******************************************",
        " ---------------------------------------------",
        f" #p {route} external='ColabUMA {model_name}/{task_name} {optimizer_name}'",
        " ---------------------------------------------",
        " ---------------",
        f" {out_prefix}",
        " ---------------",
        " Symbolic Z-matrix:",
        f" Charge = {int(charge):2d} Multiplicity = {int(multiplicity)}",
    ] + [f" {chemical_symbols[z]:<2s} {x:20.8f}{y:14.8f}{zz:14.8f}" for z, (x, y, zz) in zip(atoms0.numbers, first)]


def _gaussian_steps(r, step_suffix="", last_only=False):
    # 1回の最適化のステップを Gaussian の opt 出力と同じ順序で書く。last_only なら最終ステップだけ
    numbers = atoms0.numbers
    to_au = ase.units.Bohr / EV_PER_HARTREE  # eV/Å → Hartree/Bohr
    t = fmax * to_au
    thresholds = (t, t * 2 / 3, t * 4, t * 8 / 3)  # Gaussian の既定値と同じ比率
    dash69 = " " + "-" * 69
    dash67 = " " + "-" * 67
    grad = " GradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGradGrad"
    h = r["hist"]
    n_frames = len(h["frames"])
    out = []
    for i, (pos, e, f) in enumerate(zip(h["frames"], h["energy"], h["forces"])):
        if last_only and i < n_frames - 1:
            continue
        out += ["                          Input orientation:", dash69,
                " Center     Atomic      Atomic             Coordinates (Angstroms)",
                " Number     Number       Type             X           Y           Z", dash69]
        out += [f"{k + 1:7d}{z:11d}{0:12d}    {x:12.6f}{y:12.6f}{zz:12.6f}" for k, (z, (x, y, zz)) in enumerate(zip(numbers, pos))]
        out += [dash69, f" SCF Done:  E(UMA) = {e / EV_PER_HARTREE:17.9f}     A.U. after    1 cycles"]
        fa = f * to_au
        out += [dash67, " Center     Atomic                   Forces (Hartrees/Bohr)",
                " Number     Number              X              Y              Z", dash67]
        out += [f"{k + 1:7d}{z:9d}{fx:19.9f}{fy:15.9f}{fz:15.9f}" for k, (z, (fx, fy, fz)) in enumerate(zip(numbers, fa))]
        out += [dash67, f" Cartesian Forces:  Max {np.abs(fa).max():15.9f} RMS {np.sqrt((fa ** 2).mean()):15.9f}",
                grad, " Berny optimization.", " Search for a local minimum.",
                f" Step number{i + 1:4d} out of a maximum of{max_steps + 1:4d}{step_suffix}"]
        disp = (h["frames"][i + 1] - pos) / ase.units.Bohr if i + 1 < n_frames else np.zeros_like(pos)
        values = (np.abs(fa).max(), np.sqrt((fa ** 2).mean()), np.abs(disp).max(), np.sqrt((disp ** 2).mean()))
        out.append("         Item               Value     Threshold  Converged?")
        for label, v, th in zip(("Maximum Force", "RMS     Force", "Maximum Displacement", "RMS     Displacement"),
                                values, thresholds):
            out.append(f" {label:<21s}{v:12.6f}{th:13.6f}     {'YES' if v < th else 'NO'}")
        de = (h["energy"][i + 1] - e) / EV_PER_HARTREE if i + 1 < n_frames else 0.0
        out.append(" Predicted change in Energy=" + f"{de:.6E}".replace("E", "D"))
        if i == n_frames - 1:
            if r["converged"]:
                out += [" Optimization completed.", "    -- Stationary point found."]
            else:
                out += [" Optimization stopped.", f"    -- Number of steps exceeded,  NStep={max_steps}"]
        out.append(grad)
    return out


def _gaussian_end(converged):
    if converged:
        return [f" Normal termination of Gaussian 16 at {datetime.datetime.now():%a %b %d %H:%M:%S %Y}."]
    return [" Error termination request processed by link 9999."]


def gaussian_opt_log(r):
    out = _gaussian_header("opt", r["hist"]["frames"][0]) + [" "] + _gaussian_steps(r) + _gaussian_end(r["converged"])
    return "\n".join(out) + "\n"


def gaussian_scan_log(results):
    # 渡された各点の最終構造を Gaussian の opt=modredundant 出力と同じ書式で書く。
    # 点の番号はこのファイルの中で1から振り直す（未収束の点を除くと飛び番になるため）
    atom_ids = [i + 1 for i in scan_idx]
    name = f"{scan_code}1"
    definition = f"{scan_code}({','.join(str(i) for i in atom_ids)})"
    out = _gaussian_header("opt=modredundant", results[0]["hist"]["frames"][0])
    out += [" ",
            " The following ModRedundant input section has been read:",
            f" {scan_code.replace('R', 'B')}" + "".join(f"{i:8d}" for i in atom_ids)
            + f" S{scan_n_steps:4d}{scan_step_size:7.3f}",
            " ",
            "                           ----------------------------",
            "                           !    Initial Parameters    !",
            "                           ! (Angstroms and Degrees)  !",
            " --------------------------                            --------------------------",
            " ! Name  Definition              Value          Derivative Info.                !",
            " " + "-" * 80,
            f" ! {name:<5s} {definition:<21s} {results[0]['target']:<16.4f} Scan" + " " * 28 + "!",
            " " + "-" * 80,
            f" Number of optimizations in scan={len(results):4d}",
            " "]
    for n, r in enumerate(results, 1):
        out += _gaussian_steps(r, f" on scan point{n:6d} out of{len(results):6d}", last_only=True)
    # 各点のまとめ（5列ずつ）。エネルギーは整数部を見出しに出して差分で書く
    base = float(int(min(r["energy"] for r in results) / EV_PER_HARTREE))
    out.append(f" Summary of Optimized Potential Surface Scan (add {base:.1f} to energies):")
    for s in range(0, len(results), 5):
        part = results[s:s + 5]
        out.append(" " * 21 + "".join(f"{n:10d}" for n in range(s + 1, s + len(part) + 1)))
        out.append("     Eigenvalues --  " + "".join(f"{r['energy'] / EV_PER_HARTREE - base:10.5f}" for r in part))
        out.append(f"{name:>13s}        " + "".join(f"{r['value']:10.5f}" for r in part))
    out.append(" ")
    out += _gaussian_end(len(results) == len(scan_results))
    return "\n".join(out) + "\n"


symbols = atoms0.get_chemical_symbols()
for r in scan_results:
    r["dE"] = (r["energy"] - lowest["energy"]) / EV_PER_KCALMOL
converged_results = [r for r in scan_results if r["converged"]]

scan_xyz = "".join(
    xyz_block(symbols, r["atoms"].positions,
              f"point={r['point']} {scan_code}({','.join(str(i + 1) for i in scan_idx)})={r['value']:.4f} "
              f"energy_eV={r['energy']:.8f} energy_Ha={r['energy'] / EV_PER_HARTREE:.10f} dE_kcal_mol={r['dE']:.4f} "
              f"fmax={r['fmax']:.5f} converged={r['converged']}")
    for r in converged_results)

# --- エネルギープロファイル（Colabは日本語フォントがないため軸ラベルは英語） ---
xs = [r["target"] for r in scan_results]
ys = [r["dE"] for r in scan_results]
bad = [r for r in scan_results if not r["converged"]]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(xs, ys, marker="o", color="#1f77b4", label="Converged")
if bad:
    ax.plot([r["target"] for r in bad], [r["dE"] for r in bad], ls="", marker="o", mfc="white", color="#d62728",
            label="Not converged")
if post_index is not None:
    ax.plot(xs[post_index], ys[post_index], ls="", marker="*", ms=16, color="#ff7f0e", label="Start of post-scan opt")
if bad or post_index is not None:
    ax.legend()
ax.set_xlabel(f"{scan_xlabel}: {' '.join(str(i + 1) for i in scan_idx)}")
ax.set_ylabel("Relative energy (kcal/mol)")
ax.set_title("Relaxed scan")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

# --- 表 ---
scan_df = pd.DataFrame([{
    "点": r["point"],
    f"目標値 ({scan_unit})": round(r["target"], 4),
    f"実測値 ({scan_unit})": round(r["value"], 4),
    "エネルギー (eV)": round(r["energy"], 6),
    "エネルギー (Hartree)": round(r["energy"] / EV_PER_HARTREE, 8),
    "ΔE (kcal/mol)": round(r["dE"], 3),
    "fmax (eV/Å)": round(r["fmax"], 4),
    "ステップ数": r["steps"],
    "収束": "はい" if r["converged"] else "いいえ",
    "": " ".join(t for t, c in (("最安定", r is lowest), ("最高点", r is highest),
                                ("後の最適化の開始点", post_index == r["point"] - 1)) if c),
} for r in scan_results])
display(scan_df.style.hide(axis="index"))

print(f"最安定: 点 {lowest['point']}（{lowest['target']:.3f} {scan_unit}）  "
      f"最高点: 点 {highest['point']}（{highest['target']:.3f} {scan_unit}）  "
      f"高さ {highest['dE']:.2f} kcal/mol")

if bad:
    warn(f"最大ステップ数（{max_steps}）までに収束しなかった点が {len(bad)} 個あります："
         f"{', '.join(str(r['point']) for r in bad)}。これらの点のエネルギーは信頼できないため、"
         "scan.xyz と scan.log には入れていません（scan.log の点の番号は収束した点だけで振り直しています）。",
         "最大ステップ数を増やすか、fmax を緩めるか、刻み幅を小さくしてもう一度実行してください。")
_off = [r for r in scan_results if abs(r["deviation"]) > SCAN_TOLERANCE]
if _off:
    warn(f"目標値から {SCAN_TOLERANCE} {scan_unit} より大きくずれた点が {len(_off)} 個あります："
         f"{', '.join(str(r['point']) for r in _off)}。", "刻み幅を小さくしてもう一度実行してください。")
if scan_code == "D" and abs(abs(scan_n_steps * scan_step_size) - 360.0) < 1e-6:
    _gap = abs(scan_results[-1]["dE"] - scan_results[0]["dE"])
    if _gap > 0.5:
        warn(f"1回転して同じ角度に戻ったのに、エネルギーが {_gap:.2f} kcal/mol 違います。途中で別の配座に移った可能性があります。",
             "逆向きにもスキャンして比較するか、刻み幅を小さくしてください。")

# --- 前後の最適化の結果 ---
_free = [r for r in (pre_result, post_result) if r is not None]
if _free:
    display(pd.DataFrame([{
        "": r["label"],
        f"{scan_name} ({scan_unit})": round(r["value"], 4),
        "エネルギー (eV)": round(r["energy"], 6),
        "エネルギー (Hartree)": round(r["energy"] / EV_PER_HARTREE, 8),
        f"ΔE（{_free[0]['label']}基準, kcal/mol）": round((r["energy"] - _free[0]["energy"]) / EV_PER_KCALMOL, 3),
        "fmax (eV/Å)": round(r["fmax"], 4),
        "ステップ数": r["steps"],
        "収束": "はい" if r["converged"] else "いいえ",
    } for r in _free]).style.hide(axis="index"))
    for r in _free:
        if not r["converged"]:
            warn(f"{r['label']}は最大ステップ数（{max_steps}）までに収束しませんでした。", "最大ステップ数を増やすか、fmax を緩めてもう一度実行してください。")
if post_index == len(scan_results) - 1:
    warn("スキャンの最後の点が極小として選ばれました。スキャンの端でもエネルギーが下がり続けていた可能性があります。",
         "スキャンの範囲を広げて、極小を過ぎるところまで動かすと確実です。")
elif post_index == 0:
    warn("さかのぼって最初の点まで戻りました。スキャンの途中に極小が見つからなかったため、後の最適化は出発点の谷に戻っています。",
         "スキャンの範囲を見直してください。")

# --- 各点の構造のアニメーション ---
if converged_results:
    view = py3Dmol.view(width=640, height=420)
    view.addModelsAsFrames(scan_xyz, "xyz")
    view.setStyle({"stick": {"radius": 0.14}, "sphere": {"scale": 0.24}})
    view.setBackgroundColor("white")
    view.zoomTo()
    view.animate({"loop": "backAndForth", "interval": 150})
    view.show()

# --- 出力ファイルの作成 ---
out_dir = os.path.join(tempfile.mkdtemp(), out_prefix)
os.makedirs(out_dir)
written = {}


def save(name, text):
    p = os.path.join(out_dir, f"{out_prefix}_{name}")
    with open(p, "w", encoding="utf-8") as f:
        f.write(text)
    written[name] = p


csv_df = pd.DataFrame([{
    "point": r["point"], f"target_{scan_code}": round(r["target"], 6), f"value_{scan_code}": round(r["value"], 6),
    "energy_eV": round(r["energy"], 8), "energy_Ha": round(r["energy"] / EV_PER_HARTREE, 10),
    "dE_kcal_mol": round(r["dE"], 4), "fmax_eV_A": round(r["fmax"], 5), "steps": r["steps"], "converged": r["converged"],
} for r in scan_results])
save("scan.csv", csv_df.to_csv(index=False))
if converged_results:
    save("scan.xyz", scan_xyz)
    save("scan.log", gaussian_scan_log(converged_results))
else:
    warn("収束した点がないため、scan.xyz と scan.log は作りませんでした。")
for _tag, _r in (("preopt", pre_result), ("postopt", post_result)):
    if _r is not None:
        save(f"{_tag}.xyz", xyz_block(symbols, _r["atoms"].positions,
                                      f"{_tag} {scan_code}({','.join(str(i + 1) for i in scan_idx)})={_r['value']:.4f} "
                                      f"energy_eV={_r['energy']:.8f} energy_Ha={_r['energy'] / EV_PER_HARTREE:.10f} "
                                      f"fmax={_r['fmax']:.5f} converged={_r['converged']}"))
        save(f"{_tag}.log", gaussian_opt_log(_r))


def _v(pkg):
    return _installed(pkg) or "-"


log = [
    "ColabUMA Scan 計算ログ",
    f"日時: {datetime.datetime.now().isoformat(timespec='seconds')}",
    "",
    "[入力条件]",
    f"入力: {source_desc}",
    f"電荷: {charge}", f"スピン多重度: {multiplicity}",
    f"スキャン座標: {scan_name} {' '.join(str(i + 1) for i in scan_idx)}",
    f"開始値: {scan_targets[0]:.4f} {scan_unit}", f"刻みの回数: {scan_n_steps}", f"刻み幅: {scan_step_size} {scan_unit}",
    f"モデル: {model_name}", f"タスク: {task_name}",
    f"最適化アルゴリズム: {optimizer_name}", f"fmax: {fmax} eV/Å（{convergence}）", f"1点あたりの最大ステップ数: {max_steps}",
    f"前の最適化: {'オン' if opt_before_scan else 'オフ'}", f"後の最適化: {'オン' if opt_after_scan else 'オフ'}",
    "",
    "[ソフトウェア]",
    f"Python {platform.python_version()}", f"fairchem-core {_v('fairchem-core')}", f"ase {ase.__version__}",
    f"torch {torch.__version__}", f"numpy {np.__version__}",
    f"デバイス: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'}",
    "",
    "[結果]",
    f"最安定: 点 {lowest['point']}  E = {lowest['energy']:.8f} eV ({lowest['energy'] / EV_PER_HARTREE:.10f} Ha)",
    f"最高点: 点 {highest['point']}  E = {highest['energy']:.8f} eV ({highest['energy'] / EV_PER_HARTREE:.10f} Ha)",
]
if post_index is not None:
    log.append(f"後の最適化の開始点: 点 {post_index + 1}")
for r in _free:
    log.append(f"{r['label']}: {scan_name} = {r['value']:.4f} {scan_unit}  E = {r['energy']:.8f} eV "
               f"({r['energy'] / EV_PER_HARTREE:.10f} Ha)  fmax = {r['fmax']:.5f}  steps = {r['steps']}  converged = {r['converged']}")
for r in scan_results:
    log += ["", f"[点 {r['point']}] 目標値 = {r['target']:.4f} {scan_unit}  実測値 = {r['value']:.4f} {scan_unit}  "
                f"steps = {r['steps']}  converged = {r['converged']}",
            f"{'step':>5s} {'energy_eV':>18s} {'fmax_eV/A':>12s}"]
    log += [f"{i:5d} {e:18.8f} {fm:12.5f}" for i, (e, fm) in enumerate(zip(r["hist"]["energy"], r["hist"]["fmax"]))]
save("log.txt", "\n".join(log) + "\n")

zip_path = os.path.join(os.path.dirname(out_dir), f"{out_prefix}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in written.values():
        z.write(p, os.path.basename(p))
print("ZIPの中身:", ", ".join(os.path.basename(p) for p in written.values()))

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("ZIPを保存しました:", zip_path)